# 🏠 House Recognition — Super AI Engineer Season 6
**Binary Classification**: มีบ้านในภาพ (1) หรือไม่ (0)  
**Model**: EfficientNet-B3 (pretrained ImageNet) + Fine-tuning  
**Eval**: Accuracy Score

## 📦 Cell 1: Install & Import

In [1]:
!pip install timm -q

import os, random
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from tqdm import tqdm

print("✅ Import สำเร็จ")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")


✅ Import สำเร็จ
PyTorch version: 2.10.0+cu128
GPU available: True


## ⚙️ Cell 2: Config — แก้ค่าตรงนี้ที่เดียวพอ

In [3]:
# Mount Google Drive ก่อน
from google.colab import drive
drive.mount('/content/drive')

# ตรวจสอบว่าเห็นไฟล์ไหม
import os
base = '/content/drive/MyDrive/superai-ss6/House-recognition'
print("Files in base dir:")
print(os.listdir(base))

CFG = {
    # --- Path (ชี้ไปที่ Google Drive) ---
    'train_csv'     : f'{base}/train.csv',
    'sample_sub'    : f'{base}/sample_submission.csv',
    'train_img_dir' : f'{base}/train/train/',   # folder รูป train
    'test_img_dir'  : f'{base}/test/test/',    # folder รูป test
    'output_dir'    : f'{base}/output/',  # folder save model & submission

    # --- Model ---
    'model_name'    : 'efficientnet_b3',  # EfficientNet-B3 จาก timm
    'img_size'      : 300,                # B3 standard = 300px
    'num_classes'   : 1,                  # binary → 1 output node

    # --- Training ---
    'epochs'        : 15,
    'batch_size'    : 32,
    'num_workers'   : 2,
    'seed'          : 42,

    # --- Optimizer ---
    'lr_head'       : 1e-3,   # LR สำหรับ head (train ก่อน)
    'lr_backbone'   : 1e-4,   # LR สำหรับ backbone ตอน fine-tune
    'weight_decay'  : 1e-4,

    # --- Cross Validation ---
    'n_folds'       : 5,
    # ลด train_folds เหลือ [0] ถ้าอยากรันเร็ว
    'train_folds'   : [0, 1, 2, 3, 4],

    # --- TTA ---
    'use_tta'       : True,
    'tta_steps'     : 5,
}

os.makedirs(CFG['output_dir'], exist_ok=True)
print("\n✅ Config พร้อม")
print(f"Train CSV : {CFG['train_csv']}")
print(f"Train imgs: {CFG['train_img_dir']}")
print(f"Test imgs : {CFG['test_img_dir']}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files in base dir:
['sample_submission.csv', 'train.csv', 'train', 'test']

✅ Config พร้อม
Train CSV : /content/drive/MyDrive/superai-ss6/House-recognition/train.csv
Train imgs: /content/drive/MyDrive/superai-ss6/House-recognition/train/train/
Test imgs : /content/drive/MyDrive/superai-ss6/House-recognition/test/test/


## 🌱 Cell 3: Reproducibility & Device

In [4]:
def seed_everything(seed):
    """กำหนด seed ทุก library เพื่อให้ผลลัพธ์ reproduce ได้"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['seed'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

os.makedirs(CFG['output_dir'], exist_ok=True)


Using device: cuda


## 📊 Cell 4: Load CSV & สร้าง Fold

In [5]:
train_df = pd.read_csv(CFG['train_csv'])
sample_df = pd.read_csv(CFG['sample_sub'])

print(f"Train shape: {train_df.shape}")
print("\nClass distribution:")
print(train_df['class'].value_counts())
print(f"\nTest shape: {sample_df.shape}")

# StratifiedKFold = แต่ละ fold มี class ratio สมดุล (สำคัญมากกับ imbalanced data)
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df['class'])):
    train_df.loc[val_idx, 'fold'] = fold

print("\nFold distribution:")
print(train_df.groupby('fold')['class'].value_counts().unstack())


Train shape: (2953, 2)

Class distribution:
class
0    1520
1    1433
Name: count, dtype: int64

Test shape: (1550, 2)

Fold distribution:
class    0    1
fold           
0      304  287
1      304  287
2      304  287
3      304  286
4      304  286


## 🗂️ Cell 5: Dataset Class

In [6]:
class HouseDataset(Dataset):
    """
    Custom Dataset สำหรับโหลดรูปบ้าน
    mode='train' : โหลดรูป train พร้อม label
    mode='test'  : โหลดรูป test ไม่มี label
    """
    def __init__(self, df, img_dir, transform=None, mode='train'):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.mode      = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        if self.mode == 'train':
            img_path = os.path.join(self.img_dir, row['image_name'])
        else:
            # test: ชื่อไฟล์ = id + .jpg (เช็คให้ตรงกับ Kaggle dataset จริง)
            img_path = os.path.join(self.img_dir, row['id'] + '.jpg')

        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        if self.mode == 'train':
            label = torch.tensor(row['class'], dtype=torch.float32)
            return img, label
        else:
            return img, row['id']

print("✅ Dataset class พร้อม")


✅ Dataset class พร้อม


## 🎨 Cell 6: Transforms & Augmentation

In [7]:
# ImageNet mean/std — ใช้กับทุก pretrained model จาก timm/torchvision
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Train: มี augmentation เพื่อให้ model generalize ดีขึ้น
train_transform = transforms.Compose([
    transforms.Resize((CFG['img_size'] + 32, CFG['img_size'] + 32)),
    transforms.RandomCrop(CFG['img_size']),      # random crop บังคับไม่ยึดติด center
    transforms.RandomHorizontalFlip(p=0.5),      # พลิกซ้าย-ขวา (บ้านยังเป็นบ้านเหมือนเดิม)
    transforms.ColorJitter(                      # จำลองแสง/สีต่างกันในแต่ละวัน
        brightness=0.2, contrast=0.2,
        saturation=0.2, hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Val/Test: ไม่มี augmentation สุ่ม เพื่อให้ score stable
val_transform = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# TTA: augmentation เบาๆ สำหรับ Test Time Augmentation
tta_transform = transforms.Compose([
    transforms.Resize((CFG['img_size'] + 32, CFG['img_size'] + 32)),
    transforms.RandomCrop(CFG['img_size']),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("✅ Transforms พร้อม")


✅ Transforms พร้อม


## 🧠 Cell 7: Model (EfficientNet-B3)

In [8]:
class HouseModel(nn.Module):
    """
    EfficientNet-B3 pretrained ImageNet + custom classification head
    โครงสร้าง: backbone → Global Avg Pool → Dropout → Linear(1)
    """
    def __init__(self, model_name, pretrained=True):
        super().__init__()

        # โหลด backbone จาก timm (num_classes=0 = ตัด head ออก ได้ features แทน)
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool='avg'
        )

        n_features = self.backbone.num_features  # EfficientNet-B3 = 1536

        # Classification head
        self.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(n_features, 1),   # binary → 1 output
        )

    def forward(self, x):
        features = self.backbone(x)    # (batch, 1536)
        out = self.head(features)      # (batch, 1)
        return out.squeeze(1)          # (batch,)

# ทดสอบ model
model_test = HouseModel(CFG['model_name']).to(DEVICE)
dummy = torch.randn(2, 3, CFG['img_size'], CFG['img_size']).to(DEVICE)
out = model_test(dummy)
print(f"✅ Model output shape: {out.shape}  (ควรได้ torch.Size([2]))")
del model_test


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

✅ Model output shape: torch.Size([2])  (ควรได้ torch.Size([2]))


## 🔧 Cell 8: Training Functions

In [9]:
def train_one_epoch(model, loader, optimizer, criterion):
    """Train 1 epoch → return average loss"""
    model.train()
    total_loss = 0
    for imgs, labels in tqdm(loader, desc='Training', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def validate(model, loader, criterion):
    """Validate → return loss และ accuracy"""
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='Validating', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            total_loss += criterion(logits, labels).item()
            preds = (torch.sigmoid(logits) > 0.5).long().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy().astype(int))
    acc = (np.array(all_preds) == np.array(all_labels)).mean()
    return total_loss / len(loader), acc


def predict_tta(model, dataset, tta_steps=5):
    """
    Test Time Augmentation (TTA):
    predict หลายรอบด้วย augmentation ต่างกัน → average probability
    ได้ผลดีกว่า predict ครั้งเดียว ~1-2% accuracy
    """
    model.eval()
    all_probs, step_ids = [], []

    for step in range(tta_steps):
        dataset.transform = tta_transform
        loader = DataLoader(dataset, batch_size=CFG['batch_size'],
                            shuffle=False, num_workers=CFG['num_workers'])
        step_probs, step_ids = [], []
        with torch.no_grad():
            for imgs, ids in tqdm(loader, desc=f'TTA {step+1}/{tta_steps}', leave=False):
                probs = torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy()
                step_probs.extend(probs)
                step_ids.extend(ids)
        all_probs.append(step_probs)

    avg_probs   = np.mean(all_probs, axis=0)
    final_preds = (avg_probs > 0.5).astype(int)
    return step_ids, final_preds, avg_probs

print("✅ Training functions พร้อม")


✅ Training functions พร้อม


In [10]:
import os

train_dir = CFG['train_img_dir']
real_files = set(os.listdir(train_dir))

# เก็บเฉพาะ row ที่มีไฟล์จริงอยู่ในโฟลเดอร์
train_df = train_df[train_df['image_name'].isin(real_files)].reset_index(drop=True)
print(f"train_df หลังกรอง: {len(train_df)} rows")
print(f"Class distribution:")
print(train_df['class'].value_counts())

# สร้าง fold ใหม่บน data ที่กรองแล้ว
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df['class'])):
    train_df.loc[val_idx, 'fold'] = fold

# ทดสอบเปิดรูปแรก
from PIL import Image
test_path = CFG['train_img_dir'] + train_df['image_name'].iloc[0]
try:
    img = Image.open(test_path)
    print(f"\n✅ เปิดรูปได้! Size: {img.size}")
except FileNotFoundError:
    print(f"❌ ยังหาไม่เจอ: {test_path}")

train_df หลังกรอง: 2902 rows
Class distribution:
class
0    1500
1    1402
Name: count, dtype: int64

✅ เปิดรูปได้! Size: (640, 640)


In [11]:
from sklearn.model_selection import StratifiedKFold

# สร้าง fold ใหม่บน train_df ที่แก้ชื่อแล้ว
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df['class'])):
    train_df.loc[val_idx, 'fold'] = fold

print("✅ Fold พร้อม")
print(train_df.groupby('fold')['class'].value_counts().unstack())

# ทดสอบเปิดรูปแรกว่า path ถูกไหม
from PIL import Image
test_path = CFG['train_img_dir'] + train_df['image_name'].iloc[0]
try:
    img = Image.open(test_path)
    print(f"\n✅ เปิดรูปได้! Size: {img.size}")
    print(f"Path: {test_path}")
except FileNotFoundError:
    print(f"\n❌ ยังหาไม่เจอ: {test_path}")

✅ Fold พร้อม
class    0    1
fold           
0      300  281
1      300  281
2      300  280
3      300  280
4      300  280

✅ เปิดรูปได้! Size: (640, 640)
Path: /content/drive/MyDrive/superai-ss6/House-recognition/train/train/ChokChai4_img_13-7956791_100-6031267_a187-21598902886774_s90_y0_f90_1.jpg


## 🚀 Cell 9: Training Loop (K-Fold Cross Validation)

In [12]:
oof_preds       = np.zeros(len(train_df))   # Out-of-Fold predictions
test_probs_list = []                         # test probability จากแต่ละ fold

for fold in CFG['train_folds']:
    print(f"\n{'='*55}")
    print(f"  FOLD {fold}")
    print(f"{'='*55}")

    trn_df = train_df[train_df['fold'] != fold].reset_index(drop=True)
    val_df = train_df[train_df['fold'] == fold].reset_index(drop=True)
    print(f"Train: {len(trn_df)} | Val: {len(val_df)}")

    trn_ds = HouseDataset(trn_df,   CFG['train_img_dir'], train_transform, 'train')
    val_ds = HouseDataset(val_df,   CFG['train_img_dir'], val_transform,   'train')
    tst_ds = HouseDataset(sample_df, CFG['test_img_dir'], val_transform,   'test')

    trn_loader = DataLoader(trn_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=CFG['num_workers'])
    val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'])

    model     = HouseModel(CFG['model_name']).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()

    # ------------------------------------------------------------------
    # Phase 1: Train HEAD เท่านั้น (backbone frozen) 3 epochs
    # เหตุผล: head เป็น random weights ถ้า fine-tune ทั้งหมดเลย
    #          gradient อาจทำลาย pretrained backbone weights
    # ------------------------------------------------------------------
    for param in model.backbone.parameters():
        param.requires_grad = False

    optimizer = optim.AdamW(model.head.parameters(),
                            lr=CFG['lr_head'], weight_decay=CFG['weight_decay'])

    print("\n[Phase 1] Training HEAD only (backbone frozen)...")
    for epoch in range(3):
        trn_loss = train_one_epoch(model, trn_loader, optimizer, criterion)
        val_loss, val_acc = validate(model, val_loader, criterion)
        print(f"  Epoch {epoch+1}/3 | trn_loss: {trn_loss:.4f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f}")

    # ------------------------------------------------------------------
    # Phase 2: Unfreeze ทั้งหมด fine-tune ด้วย LR ต่ำกว่า
    # ใช้ differential LR: backbone LR ต่ำกว่า head
    # ------------------------------------------------------------------
    for param in model.backbone.parameters():
        param.requires_grad = True

    optimizer = optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': CFG['lr_backbone']},
        {'params': model.head.parameters(),     'lr': CFG['lr_head']},
    ], weight_decay=CFG['weight_decay'])

    # Cosine Annealing: LR ลดแบบ smooth → ช่วย fine-tune ได้ดีกว่า step decay
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG['epochs'], eta_min=1e-6)

    best_acc        = 0
    best_model_path = os.path.join(CFG['output_dir'], f'best_fold{fold}.pth')

    print(f"\n[Phase 2] Fine-tuning ALL layers ({CFG['epochs']} epochs)...")
    for epoch in range(CFG['epochs']):
        trn_loss = train_one_epoch(model, trn_loader, optimizer, criterion)
        val_loss, val_acc = validate(model, val_loader, criterion)
        scheduler.step()
        lr_now = optimizer.param_groups[0]['lr']
        print(f"  Epoch {epoch+1:2d}/{CFG['epochs']} | trn: {trn_loss:.4f} | val: {val_loss:.4f} | acc: {val_acc:.4f} | lr: {lr_now:.2e}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  ✅ Saved best model  (val_acc: {best_acc:.4f})")

    print(f"\n  Fold {fold} Best Val Accuracy: {best_acc:.4f}")

    # OOF Prediction
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    fold_preds = []
    with torch.no_grad():
        for imgs, _ in DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False):
            probs = torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy()
            fold_preds.extend(probs)
    oof_preds[val_df.index] = fold_preds

    # Test Prediction (TTA)
    if CFG['use_tta']:
        ids, _, probs = predict_tta(model, tst_ds, CFG['tta_steps'])
    else:
        tst_loader = DataLoader(tst_ds, batch_size=CFG['batch_size'], shuffle=False)
        ids, probs = [], []
        with torch.no_grad():
            for imgs, batch_ids in tst_loader:
                p = torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy()
                probs.extend(p); ids.extend(batch_ids)

    test_probs_list.append(np.array(probs))
    print(f"  Test predictions collected ✅")



  FOLD 0
Train: 2321 | Val: 581

[Phase 1] Training HEAD only (backbone frozen)...


  Epoch 1/3 | trn_loss: 0.5331 | val_loss: 0.4024 | val_acc: 0.8709


  Epoch 2/3 | trn_loss: 0.3913 | val_loss: 0.3297 | val_acc: 0.8847


  Epoch 3/3 | trn_loss: 0.3488 | val_loss: 0.2901 | val_acc: 0.9105

[Phase 2] Fine-tuning ALL layers (15 epochs)...


  Epoch  1/15 | trn: 0.2427 | val: 0.1496 | acc: 0.9398 | lr: 9.89e-05
  ✅ Saved best model  (val_acc: 0.9398)


...

  Epoch 12/15 | trn: 0.0072 | val: 0.2211 | acc: 0.9310 | lr: 1.05e-05


  Epoch 13/15 | trn: 0.0036 | val: 0.2145 | acc: 0.9362 | lr: 5.28e-06


  Epoch 14/15 | trn: 0.0033 | val: 0.2185 | acc: 0.9328 | lr: 2.08e-06


  Epoch 15/15 | trn: 0.0029 | val: 0.2479 | acc: 0.9276 | lr: 1.00e-06

  Fold 4 Best Val Accuracy: 0.9414


  Test predictions collected ✅


## 📈 Cell 10: OOF Score

In [13]:
oof_binary = (oof_preds > 0.5).astype(int)
oof_acc    = (oof_binary == train_df['class'].values).mean()
print(f"{'='*55}")
print(f"  Overall OOF Accuracy: {oof_acc:.4f}")
print(f"{'='*55}")
print("(ค่านี้ใกล้เคียง public leaderboard มากที่สุด)")


  Overall OOF Accuracy: 0.4249
(ค่านี้ใกล้เคียง public leaderboard มากที่สุด)


## 💾 Cell 11: สร้าง Submission CSV

In [14]:
# Ensemble: average probability จากทุก fold
final_probs = np.mean(test_probs_list, axis=0)
final_preds = (final_probs > 0.5).astype(int)

submission = pd.DataFrame({'id': ids, 'answer': final_preds})
sub_path   = os.path.join(CFG['output_dir'], 'submission.csv')
submission.to_csv(sub_path, index=False)

print(f"Submission saved → {sub_path}")
print(submission.head(10))
print("\nPrediction distribution:")
print(submission['answer'].value_counts())


Submission saved → /content/drive/MyDrive/superai-ss6/House-recognition/output/submission.csv
         id  answer
0  e4b420b0       0
1  23efa479       0
2  1f0f2402       0
3  8a60480c       0
4  11f20127       0
5  16173ced       0
6  9c05fea1       0
7  11f04229       0
8  c055fbb7       0
9  ec045077       0

Prediction distribution:
answer
0    816
1    734
Name: count, dtype: int64


## 🎯 Cell 12 (Optional): Threshold Tuning

In [ ]:
# หา threshold ที่ดีที่สุดบน OOF แทนใช้ 0.5 ตายตัว
best_threshold, best_oof_acc = 0.5, 0

for thr in np.arange(0.3, 0.7, 0.01):
    preds = (oof_preds > thr).astype(int)
    acc   = (preds == train_df['class'].values).mean()
    if acc > best_oof_acc:
        best_oof_acc   = acc
        best_threshold = thr

print(f"Best Threshold : {best_threshold:.2f}")
print(f"Best OOF Acc   : {best_oof_acc:.4f}  (vs default 0.5: {(oof_binary == train_df['class'].values).mean():.4f})")

# สร้าง submission ด้วย threshold ที่ tune แล้ว
final_preds_tuned = (final_probs > best_threshold).astype(int)
sub_tuned = pd.DataFrame({'id': ids, 'answer': final_preds_tuned})
sub_tuned_path = os.path.join(CFG['output_dir'], 'submission_tuned.csv')
sub_tuned.to_csv(sub_tuned_path, index=False)
print(f"\nTuned submission saved → {sub_tuned_path}")
